In [ ]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter


# ==========================================================
# STATE
# ==========================================================

class AgentState(TypedDict):
    query: str
    context: str
    answer: str
    relevance: str


# ==========================================================
# OPENAI SETUP
# ==========================================================

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "your-api-key"

llm = ChatOpenAI(
    model="gpt-4",
    temperature=0
)

embeddings = OpenAIEmbeddings()


# ==========================================================
# DOCUMENT INGESTION
# ==========================================================

sample_text = """
Artificial Intelligence (AI) is a branch of computer science
focused on building systems capable of performing tasks that
normally require human intelligence.

Machine Learning is a subset of AI that allows computers
to learn from data without being explicitly programmed.

Deep Learning uses neural networks with multiple layers
to solve complex tasks such as image recognition,
speech recognition, and natural language processing.

Applications of AI include chatbots, recommendation systems,
autonomous vehicles, robotics, healthcare diagnostics,
and fraud detection.
"""


# ==========================================================
# CHUNKING
# ==========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(sample_text)

print(f"Created {len(chunks)} chunks")


# ==========================================================
# VECTOR STORE
# ==========================================================

vectorstore = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)


# ==========================================================
# NODE 1 : RETRIEVE DOCUMENTS
# ==========================================================

def retrieve_documents(state: AgentState):

    print("\n--- RETRIEVING DOCUMENTS ---")

    docs = vectorstore.similarity_search(
        state["query"],
        k=3
    )

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    state["context"] = context

    print("Retrieved Context:")
    print(context[:300])

    return state


# ==========================================================
# NODE 2 : GRADE RELEVANCE
# ==========================================================

def grade_documents(state: AgentState):

    print("\n--- GRADING DOCUMENTS ---")

    prompt = f"""
You are a document relevance grader.

Question:
{state['query']}

Retrieved Context:
{state['context']}

Determine whether the retrieved context contains information
useful for answering the question.

Return ONLY:

yes

or

no
"""

    result = llm.invoke(prompt)

    grade = result.content.strip().lower()

    if "yes" in grade:
        state["relevance"] = "yes"
    else:
        state["relevance"] = "no"

    print("Relevance:", state["relevance"])

    return state


# ==========================================================
# ROUTER FUNCTION
# ==========================================================

def decide_next_step(state: AgentState):

    print("\n--- ROUTING DECISION ---")

    if state["relevance"] == "yes":
        print("Documents are relevant")
        return "generate"

    print("Documents not relevant")
    return "rewrite"


# ==========================================================
# NODE 3 : REWRITE QUERY
# ==========================================================

def rewrite_query(state: AgentState):

    print("\n--- REWRITING QUERY ---")

    prompt = f"""
You are a query optimization assistant.

Rewrite the user's question so that
a vector database can retrieve better documents.

Original Question:
{state['query']}

Return ONLY the rewritten question.
"""

    result = llm.invoke(prompt)

    new_query = result.content.strip()

    print("Old Query :", state["query"])
    print("New Query :", new_query)

    state["query"] = new_query

    return state


# ==========================================================
# NODE 4 : GENERATE ANSWER
# ==========================================================

def generate_answer(state: AgentState):

    print("\n--- GENERATING ANSWER ---")

    prompt = f"""
You are a helpful assistant.

Use ONLY the provided context
to answer the question.

Context:
{state['context']}

Question:
{state['query']}

Answer:
"""

    response = llm.invoke(prompt)

    state["answer"] = response.content

    return state


# ==========================================================
# BUILD LANGGRAPH
# ==========================================================

graph = StateGraph(AgentState)

# Add Nodes

graph.add_node(
    "retrieve",
    retrieve_documents
)

graph.add_node(
    "grade",
    grade_documents
)

graph.add_node(
    "rewrite",
    rewrite_query
)

graph.add_node(
    "generate",
    generate_answer
)

# Entry Point

graph.set_entry_point("retrieve")

# Normal Edges

graph.add_edge(
    "retrieve",
    "grade"
)

# Conditional Routing

graph.add_conditional_edges(
    "grade",
    decide_next_step,
    {
        "generate": "generate",
        "rewrite": "rewrite"
    }
)

# Loop Back

graph.add_edge(
    "rewrite",
    "retrieve"
)

# Finish

graph.add_edge(
    "generate",
    END
)


# ==========================================================
# COMPILE GRAPH
# ==========================================================

agent = graph.compile()


# ==========================================================
# VISUAL EXECUTION FLOW
# ==========================================================

"""
START
  |
  v
retrieve
  |
  v
grade
  |
  +---- yes ----> generate ----> END
  |
  +---- no -----> rewrite
                     |
                     v
                 retrieve
"""


# ==========================================================
# RUN AGENT
# ==========================================================

result = agent.invoke(
    {
        "query": "What is Artificial Intelligence?",
        "context": "",
        "answer": "",
        "relevance": ""
    }
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["answer"])